# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    # loop over tries
    for attempt in range(tries):
      # make the request
      r = requests.get('https://archive-api.open-meteo.com/v1/archive',params=params, timeout = 10, )
      print("status:", r.status_code)

      # return data frame if 200
      if r.status_code == 200:
        return pd.DataFrame(r.json()['daily'])

      # raise on status not in RETRY_STATUSES
      if r.status_code not in RETRY_STATUSES:
        raise RuntimeError("Error: no amount of tries will fix this")

      # sleep between trials
      delay = 0.5 * (2 ** attempt)
      time.sleep(delay)

    # raise if every attempt failed
    raise RuntimeError("Error: all attempts failed")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
used_fallback = False
all_weather = get_weather("2026-09-05","2026-10-03")
wx = all_weather[all_weather['time'].isin(['2026-09-05','2026-10-03'])].copy()
display(wx)

status: 200


,time,temperature_2m_max,precipitation_sum
0,2026-09-05,29.4,2.5
28,2026-10-03,22.0,14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
wx['source'] = 'open-meteo'

display(wx)

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx['time'] = pd.to_datetime(wx['time'])
wx = wx.rename(columns={
    'time':'date',
    'temperature_2m_max':'temp_max',
    'precipitation_sum':'precip_mm',
})
display(wx)

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
wettest_index = wx['precip_mm'].idxmax()
wettest_row = wx.loc[wettest_index]
difference = wx['precip_mm'].max() - wx['precip_mm'].min()

print(wettest_row['date'].date(),"was the wettest date with",wettest_row['precip_mm'],"mm of precipitation, and was wetter by",difference,"mm of precipitation")

2026-10-03 was the wettest date with 14.7 mm of precipitation, and was wetter by 12.2 mm of precipitation


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
from datetime import datetime

print("Endpoint:", BASE)
print("Coordinates:", "38.03, -78.51")
print("Date range:", "2026-09-05 to 2026-10-03")
print("Pulled:", datetime.now().strftime("%Y-%m-%d %H:%M"))

if used_fallback:
    print("Source:", "fallback sample")
else:
    print("Source:", "open-meteo")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range: 2026-09-05 to 2026-10-03
Pulled: 2026-10-09 13:22
Source: open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather() returned an empty DataFrame on failure, the weather data would be missing when it was merged with the 200 days of sales. The chart could make it look like there was no rain or leave out those rows entirely. This could go unnoticed because the code would still run, so an API failure would look like real missing data instead of an error.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# pull for orlando
orlando_all = get_weather(
    '2026-09-05',
    '2026-10-03',
    lat=28.54,
    lon=-81.38
)

# filter dates
orlando = orlando_all[
    orlando_all['time'].isin(['2026-09-05', '2026-10-03'])
].copy()

# clean data
orlando['time'] = pd.to_datetime(orlando['time'])

orlando = orlando.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

orlando['source'] = 'open-meteo'
orlando['city'] = 'Orlando'
wx['city'] = 'Charlottesville'

# combine tables - concat is a good way to stack data
both_cities = pd.concat(
    [wx, orlando],
    ignore_index=True
)

display(both_cities)

status: 200


,date,temp_max,precip_mm,source,city
0,2026-09-05,29.4,2.5,open-meteo,Charlottesville
1,2026-10-03,22.0,14.7,open-meteo,Charlottesville
2,2026-09-05,32.1,3.9,open-meteo,Orlando
3,2026-10-03,30.4,14.5,open-meteo,Orlando
